# **Install Required Libraries & Load Data**
Run this cell to install Hugging Face libraries, load the CSV, clean missing text, and balance/sample the data so training is fast and reliable.

In [ ]:
!pip install transformers accelerate -q

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Check if GPU is enabled
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load Dataset
df = pd.read_csv("Musical_instruments_reviews.csv")

# 2. Keep only the relevant columns and drop empty reviews
df = df[['reviewText', 'overall']].dropna()

# 3. Map 1-5 ratings into 3 sentiment classes
def map_sentiment(rating):
    if rating <= 2.0:
        return 0  # Negative
    elif rating == 3.0:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['overall'].apply(map_sentiment)

# 4. Take a clean sample (e.g., 3,000 reviews) so Colab finishes quickly in front of professors
# Stratify ensures all 3 classes (0, 1, 2) are represented proportionally
df_sample = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 1000), random_state=42)
).reset_index(drop=True)

print("Class distribution in sample:")
print(df_sample['label'].value_counts())

# Split into Train (80%) and Test (20%)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_sample['reviewText'].tolist(),
    df_sample['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_sample['label']
)

Using device: cpu
Class distribution in sample:
label
2    1000
1     772
0     467
Name: count, dtype: int64


/tmp/ipykernel_6517/1682736344.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('label', group_keys=False).apply(


# **Baseline Model (LSTM)**
Run this cell to build, train, and evaluate a baseline PyTorch Long Short-Term Memory (LSTM) network.

In [ ]:
# --- 1. Build a basic vocabulary ---
words = set(" ".join(train_texts).lower().split())
vocab = {word: i + 2 for i, word in enumerate(words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def text_to_tensor(text, max_len=50):
    tokens = [vocab.get(w.lower(), 1) for w in text.split()]
    tokens = tokens[:max_len] + [0] * max(0, max_len - len(tokens))
    return torch.tensor(tokens, dtype=torch.long)

class ReviewLSTMDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [text_to_tensor(t) for t in texts]
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

lstm_train_loader = DataLoader(ReviewLSTMDataset(train_texts, train_labels), batch_size=32, shuffle=True)
lstm_test_loader = DataLoader(ReviewLSTMDataset(test_texts, test_labels), batch_size=32, shuffle=False)

# --- 2. Define LSTM Architecture ---
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, num_classes=3):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hn, _) = self.lstm(embedded)
        out = self.fc(hn[-1])
        return out

lstm_model = SentimentLSTM(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.003)

# --- 3. Train Baseline LSTM ---
print("Training LSTM Baseline...")
lstm_model.train()
for epoch in range(5):
    total_loss = 0
    for x_batch, y_batch in lstm_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = lstm_model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 - Loss: {total_loss/len(lstm_train_loader):.4f}")

# --- 4. Evaluate LSTM ---
lstm_model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for x_batch, y_batch in lstm_test_loader:
        x_batch = x_batch.to(device)
        preds = lstm_model(x_batch).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

print("\n--- LSTM Baseline Results ---")
print("Accuracy:", accuracy_score(all_targets, all_preds))
print(classification_report(all_targets, all_preds, target_names=["Negative", "Neutral", "Positive"]))

Training LSTM Baseline...
Epoch 1/5 - Loss: 1.0580
Epoch 2/5 - Loss: 0.9982
Epoch 3/5 - Loss: 0.9021
Epoch 4/5 - Loss: 0.8177
Epoch 5/5 - Loss: 0.6581

--- LSTM Baseline Results ---
Accuracy: 0.34375
              precision    recall  f1-score   support

    Negative       0.23      0.25      0.24        93
     Neutral       0.33      0.53      0.40       155
    Positive       0.50      0.24      0.33       200

    accuracy                           0.34       448
   macro avg       0.35      0.34      0.32       448
weighted avg       0.38      0.34      0.34       448



# **Fine-Tuning BERT (Hugging Face)**
Run this cell to load pre-trained weights from bert-base-uncased and fine-tune the classification head on the same training set.

In [ ]:
# --- 1. Tokenization with Hugging Face ---
import logging
# Suppress warning logs from Hugging Face hub to clean up output
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class BertDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

bert_train_set = BertDataset(train_texts, train_labels)
bert_test_set = BertDataset(test_texts, test_labels)

# --- 2. Load Pre-trained BERT Model ---
bert_model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

# --- 3. Set Training Hyperparameters ---
training_args = TrainingArguments(
    output_dir="./bert_output",
    num_train_epochs=2,              # 2 epochs is sufficient for pre-trained fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,              # Low learning rate preserves pre-trained knowledge
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# --- 4. Trainer Execution ---
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_set,
    eval_dataset=bert_test_set,
    compute_metrics=compute_metrics
)

print("Fine-Tuning BERT...")
trainer.train()

# --- 5. Evaluate BERT ---
eval_results = trainer.evaluate()
print("\n--- BERT Fine-Tuned Results ---")
print(f"Final Accuracy: {eval_results['eval_accuracy']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fine-Tuning BERT...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.823912,0.811218,0.625000
2,0.731770,0.782239,0.651786


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.731770,0.782239,2,0.651786



--- BERT Fine-Tuned Results ---
Final Accuracy: 0.6518
